# Association Rule Mining using FP-Growth

This notebook demonstrates how to extract frequent itemsets and association rules from transaction data using the FP-Growth algorithm, inspired by the Kaggle notebook by Mohammed Derouiche.

---

**Dataset**: Cleaned version of the [UCI Online Retail Dataset](https://archive.ics.uci.edu/ml/datasets/online+retail)  
**Goal**: Generate association rules in the form `antecedents → consequents` to be used in a recommendation system API.


In [5]:
# Import necessary libraries
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

In [6]:
df = pd.read_csv("../data/transaction_fpgrowth.csv")

transactions = df["items"].dropna().str.split(",").tolist()

In [7]:
te = TransactionEncoder()
te_data = te.fit_transform(transactions)
df_trans = pd.DataFrame(te_data, columns=te.columns_)

In [8]:
import time
print("Running FP-Growth...")
start_time = time.time()

frequent_itemsets = fpgrowth(df_trans, min_support=0.0025, use_colnames=True, max_len=7)

print(f"FP-Growth completed in {time.time() - start_time:.2f} seconds")
print(f"Found {len(frequent_itemsets)} frequent itemsets.")


Running FP-Growth...


MemoryError: Unable to allocate 78.9 MiB for an array with shape (4107, 20136) and data type bool

In [ ]:
import re

rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1)
rules_tidy = rules[["antecedents", "consequents", "confidence"]].rename(
    columns={"antecedents": "antecedent", "consequents": "consequent"}
)

rules_tidy["antecedent"] = rules_tidy["antecedent"].apply(lambda s: next(iter(s)))
rules_tidy["consequent"] = rules_tidy["consequent"].apply(lambda s: next(iter(s)))

rules_tidy["antecedent"] = rules_tidy["antecedent"].str.lower().str.replace(r'[:,"]', '', regex=True).str.strip()
rules_tidy["consequent"] = rules_tidy["consequent"].str.lower().str.replace(r'[:,"]', '', regex=True).str.strip()

rules_tidy = rules_tidy.sort_values("confidence", ascending=False)

In [ ]:
rules_tidy.to_csv("../data/rules.csv", index=False)
print(f"Saved {len(rules_tidy)} rules to ../data/rules.csv")

Saved 0 rules to ../data/rules.csv
